<a href="https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked Actions and Reason Codes

The Week 4 baseline score is used to rank content pages for human review. It gives 50 points when a page has not been updated for at least 180 days and another 50 points when it has at least 500 impressions in the last 90 days.

The reason codes explain the ranking:
- `stale_and_visible`: High-priority human review.
- `stale_only`: Monitor content freshness.
- `visible_only`: Monitor search visibility.
- `monitor`: Routine monitoring.

Pages meeting both conditions receive 100 points and appear first. The queue is a decision-support tool, not a prediction of refresh success.

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter dataset
data_path = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

# Check the dataset
print("Dataset shape:", df.shape)
print("Number of clients:", df["client_id"].nunique())
print("Columns:", df.columns.tolist())
# Week 4 baseline scoring rule
df["stale_flag"] = df["days_since_last_update"] >= 180
df["visible_flag"] = df["impressions_90d"] >= 500

# Give 50 points for each condition
df["priority_score"] = (
    df["stale_flag"].astype(int) * 50
    + df["visible_flag"].astype(int) * 50
)

# Assign an understandable reason code
conditions = [
    df["stale_flag"] & df["visible_flag"],
    df["stale_flag"] & ~df["visible_flag"],
    ~df["stale_flag"] & df["visible_flag"]
]

reason_codes = [
    "stale_and_visible",
    "stale_only",
    "visible_only"
]

df["reason_code"] = np.select(
    conditions,
    reason_codes,
    default="monitor"
)

# Map each reason code to a suggested human action
action_map = {
    "stale_and_visible": "refresh_review",
    "stale_only": "monitor_staleness",
    "visible_only": "monitor_visibility",
    "monitor": "routine_monitoring"
}

df["suggested_action"] = df["reason_code"].map(action_map)

# Rank pages by baseline score
ranked_queue = df.sort_values(
    by=["priority_score", "days_since_last_update", "impressions_90d"],
    ascending=[False, False, False]
).copy()

ranked_queue["priority_rank"] = range(1, len(ranked_queue) + 1)

# Display the queue summary
print("Reason code counts:")
print(ranked_queue["reason_code"].value_counts())

print("\nPriority score counts:")
print(ranked_queue["priority_score"].value_counts())

print("\nTop 10 ranked actions:")
display(
    ranked_queue[
        ["priority_rank", "content_id", "priority_score",
         "reason_code", "suggested_action"]
    ].head(10)
)

Dataset shape: (30000, 44)
Number of clients: 32
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Reason code counts:
reason_code
visible_only         16709
monitor              13117
stale_only             157
stale_and_visible       17
Name: count, dtype: int64

Prio

,priority_rank,content_id,priority_score,reason_code,suggested_action
26840,1,content_7f116ae1f6f5,100,stale_and_visible,refresh_review
7452,2,content_72496874f806,100,stale_and_visible,refresh_review
16751,3,content_cf56e2e2e282,100,stale_and_visible,refresh_review
16514,4,content_7368877ea310,100,stale_and_visible,refresh_review
7021,5,content_1bfaa38ff26c,100,stale_and_visible,refresh_review
11489,6,content_5feee3994adb,100,stale_and_visible,refresh_review
698,7,content_b16bd7307b39,100,stale_and_visible,refresh_review
5327,8,content_fe16a55cd13d,100,stale_and_visible,refresh_review
26810,9,content_ecb6215e79fd,100,stale_and_visible,refresh_review
23215,10,content_bdbec75c1148,100,stale_and_visible,refresh_review


In [15]:
!pwd
!ls -la
!find /content -name "content_refresh_anonymized.csv"

/content/machine-learning-portfolio
total 104
drwxr-xr-x 12 root root  4096 Nov  8 18:05 .
drwxr-xr-x  1 root root  4096 Nov  8 18:05 ..
-rw-r--r--  1 root root   654 Nov  8 18:05 AGENTS.md
-rw-r--r--  1 root root   654 Nov  8 18:05 CLAUDE.md
drwxr-xr-x  3 root root  4096 Nov  8 18:05 data
-rw-r--r--  1 root root  2763 Nov  8 18:05 DATA_USE.md
drwxr-xr-x  3 root root  4096 Nov  8 18:05 docs
drwxr-xr-x  8 root root  4096 Nov  8 18:05 .git
drwxr-xr-x  3 root root  4096 Nov  8 18:05 .github
-rw-r--r--  1 root root   993 Nov  8 18:05 .gitignore
-rw-r--r--  1 root root 10408 Nov  8 18:05 GUIDE.md
-rw-r--r--  1 root root  1289 Nov  8 18:05 LICENSE
drwxr-xr-x  2 root root  4096 Nov  8 18:05 notebooks
drwxr-xr-x  3 root root  4096 Nov  8 18:05 outputs
-rw-r--r--  1 root root  9847 Nov  8 18:05 README.md
-rw-r--r--  1 root root   107 Nov  8 18:05 requirements.txt
drwxr-xr-x  2 root root  4096 Nov  8 18:05 scripts
-rw-r--r--  1 root root  5671 Nov  8 18:05 SETUP.md
drwxr-xr-x 14 root root  4096 

In [16]:
!git clone https://github.com/sahanabytes/machine-learning-portfolio.git

Cloning into 'machine-learning-portfolio'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 172 (delta 72), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 1.89 MiB | 7.02 MiB/s, done.
Resolving deltas: 100% (72/72), done.


In [17]:
%cd /content/machine-learning-portfolio

!ls data/raw

/content/machine-learning-portfolio
content_refresh_anonymized.csv


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


Intended Use and Limitations
The queue helps content analysts and SEO teams decide which pages to review first.

The Week 4 baseline achieved Precision@20 of 0.60 and Precision@50 of 0.60 on held-out clients. The Decision Tree achieved 0.45 and 0.56, respectively. Therefore, the baseline is selected.

The target represents observed decline, not successful refresh outcomes. The queue cannot guarantee improved traffic or rankings. Human judgment is required before any content changes.

In [18]:
# Compare the validated baseline and Decision Tree
comparison = pd.DataFrame({
    "Model": ["Week 4 Baseline", "Decision Tree"],
    "Precision@20": [0.60, 0.45],
    "Precision@50": [0.60, 0.56]
})

display(comparison)

print(
    "Selected method: Week 4 Baseline"
    "\nReason: It achieved higher Precision@20 and Precision@50 "
    "on the held-out clients."
)

,Model,Precision@20,Precision@50
0,Week 4 Baseline,0.60,0.60
1,Decision Tree,0.45,0.56


Selected method: Week 4 Baseline
Reason: It achieved higher Precision@20 and Precision@50 on the held-out clients.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Rules

Before approving an action, a person must check content accuracy, search intent, quality, visibility, business value, and update effort.

### No-Go List

- Do not publish rewritten content without human approval.
- Do not delete pages based only on a low score.
- Do not change keywords without checking search intent.
- Do not guarantee traffic growth.
- Do not automate business-critical SEO decisions.

In [20]:
# Define the checks required before a human approves an action
review_checklist = [
    "Check whether the content is outdated or inaccurate",
    "Verify that the page matches current search intent",
    "Review title, headings, and content quality",
    "Check impressions, clicks, CTR, and average position",
    "Consider business value and update effort",
    "Approve, reject, or defer the suggested action"
]

print("Human review checklist:")
for number, item in enumerate(review_checklist, start=1):
    print(f"{number}. {item}")

# Actions that must never happen automatically
no_go_actions = [
    "Publish rewritten content without human approval",
    "Delete pages based only on a low score",
    "Change keywords without checking search intent",
    "Guarantee traffic growth from a refresh score",
    "Make business-critical SEO decisions automatically"
]

print("\nActions that must NOT be automated:")
for number, item in enumerate(no_go_actions, start=1):
    print(f"{number}. {item}")

Human review checklist:
1. Check whether the content is outdated or inaccurate
2. Verify that the page matches current search intent
3. Review title, headings, and content quality
4. Check impressions, clicks, CTR, and average position
5. Consider business value and update effort
6. Approve, reject, or defer the suggested action

Actions that must NOT be automated:
1. Publish rewritten content without human approval
2. Delete pages based only on a low score
3. Change keywords without checking search intent
4. Guarantee traffic growth from a refresh score
5. Make business-critical SEO decisions automatically


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and Retraining

Review the queue when new data becomes available, freshness or visibility patterns change, model performance changes, or data quality issues appear.

Reassess the 180-day and 500-impression thresholds when necessary. Retrain the Decision Tree only when sufficient new labelled data is available. Compare it with the baseline using grouped client validation before considering replacement.

These are proposed monitoring rules, not active production monitoring.

In [21]:
# Proposed monitoring and retraining triggers
monitoring_triggers = {
    "New data": "Review the queue when new performance data becomes available",
    "Freshness changes": "Reassess the 180-day update threshold",
    "Visibility changes": "Reassess the 500-impression threshold",
    "Model performance": "Recheck Precision@20 and Precision@50 on new held-out clients",
    "Data quality": "Investigate unexpected missing values, duplicates, or schema changes"
}

print("Monitoring triggers:")
for trigger, explanation in monitoring_triggers.items():
    print(f"\n{trigger}: {explanation}")

print("\nRetraining policy:")
print("Retrain the Decision Tree only when sufficient new labelled data is available.")
print("Compare it against the baseline using grouped client validation.")
print("Replace the baseline only if a new method performs better.")

Monitoring triggers:

New data: Review the queue when new performance data becomes available

Freshness changes: Reassess the 180-day update threshold

Visibility changes: Reassess the 500-impression threshold

Model performance: Recheck Precision@20 and Precision@50 on new held-out clients

Data quality: Investigate unexpected missing values, duplicates, or schema changes

Retraining policy:
Retrain the Decision Tree only when sufficient new labelled data is available.
Compare it against the baseline using grouped client validation.
Replace the baseline only if a new method performs better.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Export Description

The notebook exports a ranked CSV queue containing priority rank, content ID, score, reason code, suggested action, update age, and impressions. The file can be regenerated by running the notebook.

The queue supports the research paper's recommendations and does not trigger automatic content changes.

In [22]:
# Create the output folder
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Select the fields needed for human review
export_columns = [
    "priority_rank",
    "content_id",
    "priority_score",
    "reason_code",
    "suggested_action",
    "days_since_last_update",
    "impressions_90d"
]

# Create the final ranked action queue
export_queue = ranked_queue[export_columns].copy()

# Export the queue as a CSV file
queue_path = output_dir / "ranked_content_action_queue.csv"
export_queue.to_csv(queue_path, index=False)

print("Export completed successfully!")
print("File:", queue_path)
print("Total content items:", len(export_queue))
print("\nPriority distribution:")
print(export_queue["priority_score"].value_counts().sort_index(ascending=False))

print("\nTop 5 recommended actions:")
display(export_queue.head(5))

Export completed successfully!
File: work/outputs/ranked_content_action_queue.csv
Total content items: 30000

Priority distribution:
priority_score
100       17
50     16866
0      13117
Name: count, dtype: int64

Top 5 recommended actions:


,priority_rank,content_id,priority_score,reason_code,suggested_action,days_since_last_update,impressions_90d
26840,1,content_7f116ae1f6f5,100,stale_and_visible,refresh_review,301,954
7452,2,content_72496874f806,100,stale_and_visible,refresh_review,301,821
16751,3,content_cf56e2e2e282,100,stale_and_visible,refresh_review,194,61678
16514,4,content_7368877ea310,100,stale_and_visible,refresh_review,194,59472
7021,5,content_1bfaa38ff26c,100,stale_and_visible,refresh_review,194,25715


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.